# Analysis of the cosmology baseline $F$ problems

## Prelude

In [ ]:
import pandas as pd
import seaborn as sns
import sympy

## Load data

In [ ]:
report = pd.read_csv("Generated/full-report.csv")

In [ ]:
report

In [ ]:
report["sympy_expr"] = report["expr_original_syms"].apply(lambda e: sympy.sympify(e))

In [ ]:
report["sympy_expr_xs"] = report["expr"].apply(lambda e: sympy.sympify(e))

## Definitions

In [ ]:
def get_data_set(data_set):
    return report[report["data_set"] == data_set]

In [ ]:
def parse_if_needed(expr_or_str) -> sympy.Expr:
    if isinstance(expr_or_str, str):
        return sympy.sympify(expr_or_str)
    elif isinstance(expr_or_str, sympy.Expr):
        return expr_or_str
    else:
        raise ValueError("Input must be a string or a sympy expression")

In [ ]:
def replace_near_integer(expr, tolerance=1e-5):
    if expr.func == sympy.Float:
        x = expr.evalf()
        x_int = round(x)
        x_frac = x - x_int
        if abs(x_frac) < tolerance:
            return sympy.Integer(x_int)
        else:
            return expr
    elif len(expr.args) == 0:
        return expr
    else:
        new_args = map(
            lambda e: replace_near_integer(e, tolerance=tolerance), expr.args
        )
        return expr.func(*new_args)

In [ ]:
def to_spiffy(expr):
    expr = sympy.expand(expr, rational=False)
    expr = replace_near_integer(expr, tolerance=1e-6)
    # Too time consuming:
    # expr = sympy.simplify(expr)
    return expr

In [ ]:
def generous_simplify(expr, tolerance=5e-3):
    expr = replace_near_integer(expr, tolerance=tolerance)
    expr = sympy.expand(expr, rational=False).evalf()
    expr = replace_near_integer(expr, tolerance=tolerance)
    expr = sympy.simplify(expr)
    expr = sympy.nsimplify(expr, tolerance=tolerance)
    return expr

In [ ]:
def count_by_threshold(df, threshold):
    return sum(df["mse"] < threshold)

In [ ]:
overall = {}

## Polynomials

Generally, the polynomial problems are easy.

### $F_1$

In [ ]:
df_f1 = get_data_set("F1").sort_values("mse")

In [ ]:
df_f1

In [ ]:
df_f1["sympy_expr_xs"].apply(generous_simplify)

After rounding, all of these are correct apart from debris.

In [ ]:
mse_threshold = 1e-11
ballpark_threshold = 5e-3

overall["F1"] = {
    "data_set": "F1",
    "perfect": 32,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_f1, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_f1, ballpark_threshold),
}

### $F_4$

In [ ]:
df_f4 = get_data_set("F4").sort_values("mse")

In [ ]:
df_f4["sympy_expr"].apply(generous_simplify)

After rounding, all of these are correct.

In [ ]:
mse_threshold = 1e-11
ballpark_threshold = 5e-3

overall["F4"] = {
    "data_set": "F4",
    "perfect": 32,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_f4, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_f4, ballpark_threshold),
}

## Rational functions

### $F_7$

In [ ]:
df_f7 = get_data_set("F7").sort_values("mse")

In [ ]:
sns.histplot(df_f7["mse"], log_scale=True)

In [ ]:
df_f7["sympy_expr"].apply(lambda e: generous_simplify(e, tolerance=5e-4))

There are 21 clearly good.

In [ ]:
mse_threshold = 1e-11
ballpark_threshold = 5e-3
overall["F7"] = {
    "data_set": "F7",
    "perfect": 17,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_f7, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_f7, ballpark_threshold),
}

## Medium

### $F_5$

In [ ]:
df_f5 = get_data_set("F5").sort_values("mse")

In [ ]:
sns.histplot(df_f5["mse"], log_scale=True)

In [ ]:
df_f5["sympy_expr"].apply(
    lambda e: generous_simplify(e, tolerance=5e-2)
)

There are 20 good ones.
A couple have  $\sin(x + \pi/2(1+\epsilon))$ instead of $\cos(x)$.

In [ ]:
mse_threshold = 1e-11
ballpark_threshold = 5e-3
overall["F5"] = {
    "data_set": "F5",
    "perfect": 20,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_f5, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_f5, ballpark_threshold),
}

## Overall

In [ ]:
pd.DataFrame(overall).T